# Lekcja 12 - Redukcja Historii Czatów za pomocą Notatnika Agenta

Ten notatnik demonstruje, jak zarządzać kontekstem w długich rozmowach przy użyciu Microsoft Agent Framework. W miarę rozwoju rozmów liczba tokenów rośnie — ostatecznie przekraczając okno kontekstowe modelu. Rozwiązujemy to za pomocą **wzoru podsumowywania kontekstu** oraz **notatnika agenta** do trwałej pamięci.

## Czego się nauczysz:
1. **Dlaczego zarządzanie kontekstem ma znaczenie**: Zrozumienie limitów tokenów i okien kontekstowych
2. **Agenci świadomi kontekstu**: Tworzenie agentów, którzy zarządzają własnym kontekstem rozmowy
3. **Wzór podsumowywania kontekstu**: Wykorzystanie narzędzi do kondensacji historii rozmowy
4. **Notatnik agenta**: Trwała pamięć, która przetrwa redukcję kontekstu

## Wymagania wstępne:
- Konfiguracja Azure OpenAI z ustawionymi zmiennymi środowiskowymi
- Zrozumienie podstawowych koncepcji agentów z poprzednich lekcji


## Konfiguracja


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity --quiet

In [ ]:
import os
import asyncio
from datetime import datetime
from pathlib import Path

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Load environment variables and create Azure AI Foundry provider
load_dotenv()

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ Azure AI Foundry provider configured")


## Dlaczego zarządzanie kontekstem ma znaczenie

Każdy LLM ma skończone **okno kontekstu** — maksymalną liczbę tokenów, które może przetworzyć w pojedynczym żądaniu. W miarę postępu rozmowy wieloetapowej:

- **Liczba tokenów rośnie liniowo** z każdą wiadomością użytkownika i odpowiedzią asystenta.
- **Tokeny w promptcie dominują koszty**, ponieważ cała historia jest wysyłana ponownie przy każdym kroku.
- Ostatecznie rozmowa **przekracza okno kontekstu**, a model albo ucinana, albo generuje błąd.

### Strategie zarządzania kontekstem

| Strategia | Jak działa | Kompromis |
|---|---|---|
| **Ucinanie** | Usunięcie najstarszych wiadomości | Utrata wczesnego kontekstu |
| **Podsumowanie** | Skrócenie starszych wiadomości do streszczenia | Część szczegółów utracona, ale kluczowe punkty zachowane |
| **Scratchpad / Pamięć zewnętrzna** | Przechowywanie kluczowych faktów poza rozmową | Wymaga wywołań narzędzi, ale przetrwa każde ograniczenie |

W tym notatniku łączymy **podsumowanie** z narzędziem **scratchpad**, aby agent mógł utrzymać ciągłość nawet wtedy, gdy historia rozmowy jest skrócona.


## Tworzenie agenta świadomego kontekstu


In [ ]:
agent = await provider.create_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("🤖 Context-aware travel planning agent created")

## Symulacja długiej rozmowy

Przeanalizujmy rozmowę wieloetapową, aby zobaczyć, jak akumuluje się kontekst. Agent powinien zachować kluczowe szczegóły (preferencje, budżet, daty podróży) w kolejnych turach i wykazać ciągłość.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Zauważ, jak agent zachowuje kontekst z wcześniejszych wypowiedzi — wie o Japonii, sushi, świątyniach, fotografii, budżecie 3000 dolarów, podróży solo i terminie w kwietniu. W krótkiej rozmowie działa to dobrze, ale wraz z rozwojem rozmowy pełna historia staje się kosztowna do ponownego wysyłania.

Kontynuujmy rozmowę kolejnymi wypowiedziami, aby zobaczyć gromadzenie się kontekstu:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Wzorzec podsumowania kontekstu

W miarę rozwoju rozmowy możemy użyć **narzędzia do podsumowywania**, aby skondensować zgromadzony kontekst do zwartego formatu. Agent wywołuje to narzędzie, aby zapisać kluczowe preferencje, tak aby nawet jeśli starsze wiadomości zostaną usunięte, podstawowe informacje zostały zachowane.

Ten wzorzec stanowi podstawę do bardziej zaawansowanego redukowania historii:
1. Agent identyfikuje kluczowe fakty z rozmowy
2. Wywołuje narzędzie do podsumowywania, aby je utrwalić
3. Starsze wiadomości mogą być bezpiecznie usunięte, ponieważ podsumowanie zawiera najważniejsze informacje

Poniżej definiujemy narzędzie `summarize_preferences`, które agent może wywołać, aby zapisać zwarty zbiór tego, czego się dowiedział.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = await provider.create_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("🤖 Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Podsumowanie

W tej lekcji nauczyłeś się, jak zarządzać kontekstem w długotrwałych rozmowach agentów za pomocą Microsoft Agent Framework:

### Kluczowe pojęcia
- **Okna kontekstowe są skończone** — każdy token w historii rozmowy kosztuje pieniądze i wlicza się do limitu.
- **Narzędzia do podsumowań** pozwalają agentowi skondensować zgromadzony kontekst do zwięzłych streszczeń, zmniejszając wykorzystanie tokenów przy zachowaniu istotnych informacji.
- **Notatniki agenta** zapewniają trwałą zewnętrzną pamięć, która przetrwa każde ograniczenie rozmowy.

### Co zbudowałeś
- **Agenta świadomego kontekstu**, który utrzymuje ciągłość w wieloetapowych rozmowach
- **Narzędzie do podsumowań** (`summarize_preferences`), które zapisuje kluczowe dane użytkownika w zwartej formie
- **Wieloetapową rozmowę** demonstrującą zatrzymywanie i obsługę zmian kontekstu

### Zastosowania w praktyce
- **Boty obsługi klienta**: zapamiętują preferencje podczas długich sesji wsparcia
- **Asystenci osobisty**: śledzą bieżące projekty bez konieczności ponownego wyjaśniania kontekstu
- **Nauczyciele edukacyjni**: utrzymują postępy ucznia w wielu interakcjach

### Kolejne kroki
- Zaimplementuj pełne narzędzie notatnika z trwałością opartą na plikach
- Dodaj automatyczne skracanie historii po podsumowaniu
- Połącz z bazami danych wektorów dla semantycznego wyszukiwania pamięci
- Zbuduj agentów, którzy mogą wznowić rozmowy po kilku dniach z pełnym kontekstem


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Zrzeczenie się odpowiedzialności**:
Niniejszy dokument został przetłumaczony za pomocą usługi tłumaczenia AI [Co-op Translator](https://github.com/Azure/co-op-translator). Pomimo naszych starań o dokładność, prosimy mieć na uwadze, że automatyczne tłumaczenia mogą zawierać błędy lub nieścisłości. Oryginalny dokument w języku źródłowym należy traktować jako wiarygodne źródło informacji. W przypadku informacji krytycznych zalecane jest skorzystanie z profesjonalnego tłumaczenia wykonanego przez człowieka. Nie ponosimy odpowiedzialności za jakiekolwiek nieporozumienia lub błędne interpretacje wynikające z korzystania z tego tłumaczenia.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
